# AWS Bedrock AgentCore — LangGraph Agent

This notebook deploys a LangGraph ReAct agent to Bedrock AgentCore Runtime using the **HTTP protocol**. The agent has **no local tools** — it connects to the AgentCore-hosted MCP math server and uses its async tools via `langchain-mcp-adapters`.

### Architecture
```
Client → AgentCore (HTTP) → LangGraph Agent → AgentCore (MCP) → MCP Math Server
```

### Tests
- Invoke the agent in parallel with unique sessions
- Invoke the agent in parallel with a shared session
- Compare latency and server instance distribution

## Install Dependencies

In [1]:
#!pip install --force-reinstall -U -r requirements.txt --quiet

## Setup

In [2]:
from bedrock_agentcore_starter_toolkit import Runtime
from bedrock_agentcore_starter_toolkit.operations.runtime import destroy_bedrock_agentcore
from boto3.session import Session
from pathlib import Path
import os

boto_session = Session()
region = boto_session.region_name

agentcore_control_client = boto_session.client("bedrock-agentcore-control", region_name=region)
ssm_client = boto_session.client("ssm", region_name=region)

agent_name = "blog_langgraph_agent"

print(f"Using AWS region: {region}")

/Users/mingyyzz/dev-local/aws.blog1/.venv/lib/python3.14/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


Using AWS region: us-west-2


## Deploy AgentCore LangGraph Agent

Configure and launch the LangGraph agent using the **HTTP** protocol. The agent exposes a `POST /invocations` endpoint that accepts `{"prompt": "..."}` payloads. It connects to the MCP math server to use its async tools.

In [5]:
# Configure and launch the LangGraph agent
agentcore_runtime = Runtime()
response = agentcore_runtime.configure(
    entrypoint="agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    region=region,
    protocol="HTTP",
    agent_name=agent_name,
)

launch_result = agentcore_runtime.launch()
print(f"Launch completed. Agent ARN: {launch_result.agent_arn}")

# Store the agent ARN in Parameter Store for the test script
ssm_client.put_parameter(
    Name="/blog_langgraph_agent/runtime_iam/agent_arn",
    Value=launch_result.agent_arn,
    Type="String",
    Description="Agent ARN for blog_langgraph_agent",
    Overwrite=True,
)

Entrypoint parsed: file=/Users/mingyyzz/dev-local/aws.blog1/src/langgraph_agent/agent.py, bedrock_agentcore_name=agent
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: blog_langgraph_agent
Memory disabled
Network mode: PUBLIC


📄 Using existing Dockerfile: /Users/mingyyzz/dev-local/aws.blog1/src/langgraph_agent/Dockerfile

Generated .dockerignore: /Users/mingyyzz/dev-local/aws.blog1/src/langgraph_agent/.dockerignore
Keeping 'blog_langgraph_agent' as default agent
Bedrock AgentCore configured: /Users/mingyyzz/dev-local/aws.blog1/src/langgraph_agent/.bedrock_agentcore.yaml
🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'blog_langgraph_agent' to account 980413094772 (us-west-2)
Generated image tag: 20260331-231127-025
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: blog_langgraph_agent
ECR repository available: 980413094772.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-blog_langgraph_agent
Gett

✅ Reusing existing ECR repository: 980413094772.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-blog_langgraph_agent


✅ Reusing existing execution role: arn:aws:iam::980413094772:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-13dd8d44ac
Execution role available: arn:aws:iam::980413094772:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-13dd8d44ac
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: blog_langgraph_agent
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-13dd8d44ac
Reusing existing CodeBuild execution role: arn:aws:iam::980413094772:role/AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-13dd8d44ac
Using dockerignore.template with 47 patterns for zip filtering
Uploaded source to S3: blog_langgraph_agent/source.zip
Updated CodeBuild project: bedrock-agentcore-blog_langgraph_agent-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.0s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 8.4s
🔄 DOWNLOAD_SOURCE started (total: 

Launch completed. Agent ARN: arn:aws:bedrock-agentcore:us-west-2:980413094772:runtime/blog_langgraph_agent-79UgbT8kD3


{'Version': 2,
 'Tier': 'Standard',
 'ResponseMetadata': {'RequestId': 'c6796d39-0d80-44c4-ae4f-22f984323eb7',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'server': 'Server',
   'date': 'Tue, 31 Mar 2026 23:12:04 GMT',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '31',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'c6796d39-0d80-44c4-ae4f-22f984323eb7',
   'cache-control': 'no-store'},
  'RetryAttempts': 0}}

## Test: 5 Calls, Unique Sessions

Each call gets its own session on the LangGraph agent, so each is routed to a separate warm pool instance. The agent internally calls the MCP server's async tools.

In [ ]:
!python test_agent.py --calls 5 --session unique

## Test: 15 Calls, Unique Sessions — Cold Start Beyond Warm Pool

With 15 concurrent requests exceeding the warm pool of 10:
- First 10 served immediately from the warm pool
- Remaining 5 trigger cold starts with additional latency

In [ ]:
!python test_agent.py --calls 15 --session unique

## Test: 5 Calls, Shared Session

All calls share the same session, routing to a single agent instance. Tests whether the async event loop can handle concurrent requests on one VM, including concurrent MCP calls to the math server.

In [ ]:
!python test_agent.py --calls 5 --session shared

## Analysis

Key observations:
- **Agent → MCP chain**: Each agent invocation triggers an LLM call (Claude via Bedrock) plus an MCP tool call, so latency includes both LLM inference + MCP round-trip + tool execution
- **Same session behavior**: Session-to-VM mapping is identical to MCP — shared sessions pin to one instance, unique sessions fan out across the warm pool
- **Warm pool applies equally**: The 10-instance warm pool works the same for HTTP protocol agents as for MCP servers

## Cleanup (Optional)

In [ ]:
# Uncomment to clean up resources
# try:
#     ssm_client.delete_parameter(Name="/blog_langgraph_agent/runtime_iam/agent_arn")
#     print("Parameter Store parameter deleted")
# except ssm_client.exceptions.ParameterNotFound:
#     print("Parameter Store parameter not found")

# destroy_bedrock_agentcore(
#     config_path=Path(".bedrock_agentcore.yaml"),
#     agent_name=agent_name,
#     delete_ecr_repo=True,
# )